In [1]:
from transformers import BartTokenizer
from datasets import Dataset
from transformers import BartForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import torch
from scripts.Utils import TimexNorm_Utils
from scripts.Reader import obtain_combined_dataset

tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
utils = TimexNorm_Utils(tokenizer)

In [2]:
tokenizer.add_special_tokens({"additional_special_tokens": ["<timex","type=DATE>","type=TIME>","type=DURATION>","type=SET>","</timex>", "<sep>"]})
model.resize_token_embeddings(len(tokenizer))

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


BartScaledWordEmbedding(50272, 768, padding_idx=1)

In [3]:
datasets = obtain_combined_dataset(["TempEval3","wikiwars","tweets"], "normalised")

In [4]:
datasets = utils.tokenize_datasets(datasets)

In [5]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results/TimeNormBart",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    num_train_epochs=15,
    predict_with_generate=True,
    eval_strategy="steps",
    save_strategy="steps",
    logging_steps=500,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["eval"],
    data_collator=data_collator,
    compute_metrics=utils.compute_metrics,
)

In [6]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy strict,Accuracy relaxed
500,0.696500,0.337674,0.481339,0.510428
1000,0.153400,0.348263,0.529638,0.557629
1500,0.135900,0.333538,0.551043,0.578485
2000,0.119300,0.374423,0.534029,0.558727
2500,0.113600,0.348515,0.538419,0.566411
3000,0.100400,0.384665,0.509330,0.528540
3500,0.095300,0.364960,0.527991,0.547201
4000,0.086600,0.375556,0.515917,0.540615
4500,0.074100,0.383731,0.569155,0.586718
5000,0.074500,0.391917,0.539517,0.566411


d:\GeoTKG\venv\Lib\site-packages\transformers\modeling_utils.py:3854: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=29715, training_loss=0.043759636062174034, metrics={'train_runtime': 5152.8468, 'train_samples_per_second': 46.119, 'train_steps_per_second': 5.767, 'total_flos': 7.118685773402112e+16, 'train_loss': 0.043759636062174034, 'epoch': 15.0})

In [10]:
trainer.save_model("./results/TimeNormBart")
tokenizer.save_pretrained("./results/TimeNormBart")

('./results/TimeNormBart\\tokenizer_config.json',
 './results/TimeNormBart\\special_tokens_map.json',
 './results/TimeNormBart\\vocab.json',
 './results/TimeNormBart\\merges.txt',
 './results/TimeNormBart\\added_tokens.json')

In [9]:
trainer.evaluate(datasets["test"])

{'eval_loss': 0.3887539803981781,
 'eval_accuracy strict': 0.6441136671177267,
 'eval_accuracy relaxed': 0.6508795669824087,
 'eval_runtime': 11.5497,
 'eval_samples_per_second': 63.984,
 'eval_steps_per_second': 4.069,
 'epoch': 15.0}